# Fetch Data

Populates all data repositories from external sources. Run this notebook once
before `train.ipynb`. Each section is independent — re-run only the sections
you need to refresh.

**Sections (in order):**
1. Prices — daily OHLCV bars from Alpaca
2. Fundamentals — P/E, ROE, beta etc. from Yahoo Finance
3. News import — bulk-import from a Kaggle CSV into the article store
4. Coverage screen — filter the universe by avg articles/month → `data/symbols.yml`
5. Sentiment encoding — run FinBERT on screened symbols → `data/sentiment/`

**Prerequisites:**
- `.env` with `ALPACA_API_KEY` / `ALPACA_API_SECRET`
- Kaggle CSV downloaded to the path set in `KAGGLE_CSV_PATH` (section 3)

In [ ]:
from __future__ import annotations

import logging
from datetime import datetime
from pathlib import Path

import yaml

import src  # triggers load_dotenv()
from src.log import setup_logging

setup_logging()
logger = logging.getLogger("fetch_data")

## Config

In [ ]:
# Candidate ticker universe — all symbols attempted for price/fundamental fetching.
# The coverage screen (section 4) will narrow this down to symbols with sufficient
# news coverage and write the result to SYMBOLS_PATH.
CANDIDATE_SYMBOLS_PATH = Path("../data/ru1000.yml")  # {companies: {TICKER: Name}}
SYMBOLS_PATH           = Path("../data/symbols.yml")            # written by section 4

# Alpaca: years to fetch (one CSV per symbol per year)
PRICE_YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# Coverage screen thresholds
COVERAGE_START   = (2018, 1)  # (year, month) inclusive
COVERAGE_END     = (2025, 12)
MIN_AVG_ARTICLES = 5.0        # avg articles per calendar month

from src.training import ComputeConfig
compute_config = ComputeConfig()  # auto-detects cuda / cpu
print(f"Device: {compute_config.device}")

candidate_symbols: list[str] = list(
    yaml.safe_load(CANDIDATE_SYMBOLS_PATH.read_text())["companies"].keys()
)
print(f"Candidate universe: {len(candidate_symbols)} symbols")

## 1 — Prices

Fetches daily OHLCV bars from Alpaca for each symbol × year and stores them
under `data/historical-prices/<SYMBOL>/<YEAR>.csv`. Symbols already on disk
are skipped.

In [ ]:
from src.providers.alpaca import AlpacaProvider
from src.repositories.prices import PriceRepository

alpaca     = AlpacaProvider()
price_repo = PriceRepository()

stored = skipped = 0
for symbol in candidate_symbols:
    for year in PRICE_YEARS:
        try:
            price_repo.load(symbol, year)  # raises if missing
            skipped += 1
            continue
        except FileNotFoundError:
            pass

        df = alpaca.fetch_bars(
            symbols=[symbol],
            timeframe="1Day",
            start=datetime(year, 1, 1),
            end=datetime(year, 12, 31),
        )
        if df.empty:
            logger.warning("%s %d: no bars returned", symbol, year)
            continue
        price_repo.store(symbol, year, df[df["symbol"] == symbol].drop(columns="symbol"))
        stored += 1

print(f"Stored: {stored}  |  Already on disk (skipped): {skipped}")

## 4 — Coverage Screen

Filters the candidate universe to symbols with sufficient news coverage, then
writes the passing symbols to `data/symbols.yml`. This file is the screened
universe consumed by `train.ipynb`.

In [ ]:
from src.features.screening import screen_by_coverage
from src.repositories.articles import ArticleRepository

article_repo = ArticleRepository()

coverage_df = screen_by_coverage(
    repository=article_repo,
    tickers=candidate_symbols,
    start=COVERAGE_START,
    end=COVERAGE_END,
    min_avg_articles=MIN_AVG_ARTICLES,
)

passing = coverage_df[coverage_df["passes"]]
print(f"Passing: {len(passing)} / {len(candidate_symbols)} symbols")
print(coverage_df.sort_values("avg_articles_per_month", ascending=False).head(10).to_string(index=False))

In [ ]:
# Load candidate name mapping to preserve names in the output file
candidate_names: dict[str, str] = yaml.safe_load(
    CANDIDATE_SYMBOLS_PATH.read_text()
)["companies"]

screened = {
    row["ticker"]: candidate_names.get(row["ticker"], "")
    for _, row in passing.iterrows()
}

SYMBOLS_PATH.parent.mkdir(parents=True, exist_ok=True)
SYMBOLS_PATH.write_text(
    yaml.dump({"companies": screened}, default_flow_style=False, allow_unicode=True)
)
print(f"Saved {len(screened)} symbols to {SYMBOLS_PATH}")

## 5 — Sentiment Encoding

Runs the two-step NLP pipeline (summarization → FinBERT) on all articles for
each screened symbol and stores daily aggregates under `data/sentiment/`.

This is the most expensive section — expect several hours on CPU for a large
universe. Symbols already in `SentimentRepository` are skipped so the cell is
safe to re-run after interruption.

In [ ]:
from src.embeddings.pipeline import SentimentPipeline, aggregate_daily
from src.repositories.articles import ArticleRepository
from src.repositories.sentiment import SentimentRepository

screened_symbols: list[str] = list(
    yaml.safe_load(SYMBOLS_PATH.read_text())["companies"].keys()
)

article_repo = ArticleRepository()
sent_repo    = SentimentRepository()
pipeline     = SentimentPipeline(device=compute_config.device)

for i, symbol in enumerate(screened_symbols):
    if sent_repo.exists(symbol):
        logger.info("[%d/%d] %s — already encoded, skipping", i + 1, len(screened_symbols), symbol)
        continue

    logger.info("[%d/%d] %s — loading articles", i + 1, len(screened_symbols), symbol)
    articles = article_repo.load_for_ticker(symbol)
    if not articles:
        logger.warning("%s: no articles found", symbol)
        continue

    encodings = pipeline.encode_articles(articles)
    daily_df  = aggregate_daily(articles, encodings, symbol)
    sent_repo.store(symbol, daily_df)
    logger.info("%s: stored %d daily rows", symbol, len(daily_df))

print("Sentiment encoding complete.")